# Gitea Flixtube Repository

In this notebook, we look at how we can set up a monorepo on Gitea for `Flixtube`.

---

## Ensure a Default Tea Login to Gitea Exists
- Make sure you have added a login to Tea and set it as the default login.
  - If no login is listed below, use this code to create one.
    - Be sure to replace the token below with YOUR token for John.

      ```bash
      tea login add -n john.doe -u http://192.168.122.99:3000 -t 19856996a7e498aebd8c51760c0ff889b940503c -i
      tea login default john.doe
      ```

In [1]:
%%bash

tea login ls

┌──────────┬────────────────────────────┬─────────────────────┬──────────┬─────────┐
│   NAME   │            URL             │      SSH HOST       │   USER   │ DEFAULT │
├──────────┼────────────────────────────┼─────────────────────┼──────────┼─────────┤
│ john.doe │ http://192.168.122.99:3000 │ 192.168.122.99:3000 │ john.doe │ true    │
└──────────┴────────────────────────────┴─────────────────────┴──────────┴─────────┘


---

## Create a `flixtube` Organization on Gitea

- Run the command below to create a Gitea Organization for Flixtube.

In [2]:
%%bash

tea organization create flixtube


  # flixtube                                                                  
                                                                              
  • Visibility: public                                                        
  • Location:                                                                 
  • Website:                                                                  



---

## Create a `flixtube` Repository

- Run the command below to create a Gitea Repository for Flixtube.

In [3]:
%%bash

tea repos create \
    --name flixtube \
    --owner flixtube \
    --description Flixtube Repository \
    --init \
    --labels Advanced \
    --gitignores VisualStudio VisualStudioCode \
    --license GPL-3.0 \
    --readme Default \
    --branch main \
    # --private


  # flixtube/flixtube                                                         
                                                                              
  *Flixtube*                                                                  
                                                                              
  Issues: 0, Stars: 0, Forks: 0, Size: 44 KB                                  
                                                                              
  Updated: 2026-03-19 21:21 (0s ago)                                          
                                                                              
  • Browse:	http://192.168.122.99:3000/flixtube/flixtube                       
  • Clone:	git@localhost:flixtube/flixtube.git                                 
  • Permission:	admin                                                          

http://192.168.122.99:3000/flixtube/flixtube


- List Gitea repositories to make sure it was created.

In [5]:
%%bash

tea repos ls

┌──────────┬──────────┬────────┬─────────────────────────────────────┐
│  OWNER   │   NAME   │  TYPE  │                 SSH                 │
├──────────┼──────────┼────────┼─────────────────────────────────────┤
│ flixtube │ flixtube │ source │ git@localhost:flixtube/flixtube.git │
│ MyOrg    │ testrepo │ source │ git@localhost:MyOrg/testrepo.git    │
└──────────┴──────────┴────────┴─────────────────────────────────────┘


- Clone the repository to the folder `flixtube` using SSH.
  - We are also adding the `user.name` and `user.email` for John Doe to the local Git configuration (so that John is displayed for any pushed commits to Gitea).

In [6]:
%%bash

# git clone http://192.168.122.99:3000/MyOrg2/flixtube/flixtube.git flixtube
git clone git@192.168.122.99:flixtube/flixtube.git flixtube

cd flixtube

git config user.name John Doe
git config user.email john.doe@gmail.com

Cloning into 'flixtube'...


- Copy the contents of the folder `../../flixtube` into the folder `flixtube`

In [7]:
%%bash

cp -r ../../flixtube/. ./flixtube/

- Commit and push all changes to Gitea.

In [8]:
%%bash

cd flixtube
git add .
git commit --quiet -m "initial commit"
git push

remote: . Processing 1 references        
remote: Processed 1 references in total        
To 192.168.122.99:flixtube/flixtube.git
   a554deb..53d76f0  main -> main


---

## Verify the Gitea Repository was Created

- Visit http://192.168.122.99:3000/flixtube/flixtube
  - Notice the new public `flixtube` repository.
  - Notice all the code for Flixtube is available in the `flixtube` repository.
- `.github/workflows` contains all the Gitea Actions workflows.
  - The workflows make up Flixtube's CI/CD pipeline.
  - In this case, there is one CI and one CD workflow per microservice.
  - There are also 3 CD workflows for the backing services (RabbitMQ, SQLServer, Minio).
- `Flixtube.AzureStorage` contains the `video-storage` microservice for communicating with an Azure Storage Container.
- `Flixtube.Gateway` contains the `gateway` microservice for redirecting HTTP Requests to the other backend microservices.
- `Flixtube.History` contains the `history` microservice for managing video viewing history.
- `Flixtube.Metadata` contains the `metadata` microservice for managing video metadata.
- `Flixtube.MinioStorage` contains the `video-storage` microservice for communicating with a Minio Object Store.
- `Flixtube.VideoStreaming` contains the `video-streaming` microservice for streaming videos.
- `Flixtube.VideoUpload` contains the `video-upload` microservice for uploading videos.
- `Flixtube.Web` contains the `web` microservice for providing a web-based user interface for Flixtube.
- `compose` contains all the Docker Compose YAML files for bringing up a microservice with all its dependencies (this is used when running tests in the CI pipeline).
- `fixtures` contains bash scripts for Minio and a sample video.
- `scripts` contains Kubernetes YAML manifest files and bash scripts for building, publishing and deploying images, and for running tests (both to an AKS cluster and to a local K8S cluster).
- `terraform` contains Terraform HCL files for brining up and tearing down the necessary infrastructure for Flixtube on Azure.
- `Flixtube.sln` is the Solution file for Flixtube with references to all the included projects (`.csproj`).
  - This can be excluded, especially when not using a monorepo.
- `.gitignore`, `LICENSE` and `README.md` are the *usual suspects* found in a Gitea repository. 

---

## Set Variables and Secrets in Gitea

Gitea Variables and Secrets can be used in Gitea Actions workflows.
- Gitea Variables have unencrypted values.
- Gitea Secrets have encrypted values.
- Gitea Variables and Secrets can be set on an Organization level, or on a Repository level.
  - If a Variable or Secret with the same name is set both on an Organization level and a Repository level, the Repository setting overrides the Organization setting.

Set Respository Variables:
- Visit http://192.168.122.99:3000/flixtube/flixtube, click `Settings` (top right), and choose `Actions` $\rightarrow$ `Variables`.
  - You can also reach the repository variables by visiting http://192.168.122.99:3000/flixtube/flixtube/settings/actions/variables
- Click the `Add Variable` button.
  - Fill in:
    - Name: `CONTAINER_REGISTRY_REPOSITORY`
    - Value: `flixtube`
    - Description: `The Repository for the Gitea container registry.`
  - Click the `Confirm` button.

Set Organization Variables:
- Visit http://192.168.122.99:3000/flixtube, click `Settings` (top right), and choose `Actions` $\rightarrow$ `Variables`.
  - You can also reach the organization variables by visiting http://192.168.122.99:3000/org/flixtube/settings/actions/variables
- Click the `Add Variable` button.
  - Fill in:
    - Name: `CONTAINER_REGISTRY_ORGANIZATION`
    - Value: `flixtube`
    - Description: `The Organization of the Gitea container registry.`
  - Click the `Confirm` button.
- Click the `Add Variable` button.
  - Fill in:
    - Name: `CONTAINER_REGISTRY_IP`
    - Value: `192.168.122.99`
    - Description: `The IP address of the Gitea container registry.`
  - Click the `Confirm` button.
- Click the `Add Variable` button.
  - Fill in:
    - Name: `CONTAINER_REGISTRY_PORT`
    - Value: `3000`
    - Description: `The Port number of the Gitea container registry.`
  - Click the `Confirm` button.

Set Organization Secrets:
- Visit http://192.168.122.99:3000/flixtube, click `Settings` (top right), and choose `Actions` $\rightarrow$ `Secrets`.
  - You can also reach the organization secrets by visiting http://192.168.122.99:3000/org/flixtube/settings/actions/secrets
- Click the `Add Secret` button.
  - Fill in:
    - Name: `CONTAINER_REGISTRY_USERNAME`
    - Value: `john.doe`
    - Description: `The Username for the Gitea container registry.`
  - Click the `Confirm` button.

- Click the `Add Secret` button.
  - Fill in:
    - Name: `CONTAINER_REGISTRY_PASSWORD`
    - Value: `19856996a7e498aebd8c51760c0ff889b940503c`
      - **Note:** Replace this token with YOUR token for user `john.doe`.
    - Description: `The Password for the Gitea container registry.`
  - Click the `Confirm` button.

- Click the `Add Secret` button.
  - Fill in:
    - Name: `BASE64_KUBE_CONFIG`
    - Value: `...`
      - **Note:** Replace this value with your Base64-encoded Kubernetes configuration.
      - You can get this value by running the cell below.
    - Description: `The base64-encoded Kubernetes configuration file.`
  - Click the `Confirm` button.

In [12]:
%%bash

cat ~/.kube/config-u24s02-base64

YXBpVmVyc2lvbjogdjEKY2x1c3RlcnM6Ci0gY2x1c3RlcjoKICAgIGNlcnRpZmljYXRlLWF1dGhv
cml0eS1kYXRhOiBMUzB0TFMxQ1JVZEpUaUJEUlZKVVNVWkpRMEZVUlMwdExTMHRDazFKU1VSQ1ZF
TkRRV1V5WjBGM1NVSkJaMGxKUzNGTWQzRktUbUpPSzAxM1JGRlpTa3R2V2tsb2RtTk9RVkZGVEVK
UlFYZEdWRVZVVFVKRlIwRXhWVVVLUVhoTlMyRXpWbWxhV0VwMVdsaFNiR042UVdWR2R6QjVUbXBC
ZWsxVWEzaFBWRWwzVGtSV1lVWjNNSHBPYWtGNlRWUlplRTlVU1RGT1JGWmhUVUpWZUFwRmVrRlNR
bWRPVmtKQlRWUkRiWFF4V1cxV2VXSnRWakJhV0UxM1oyZEZhVTFCTUVkRFUzRkhVMGxpTTBSUlJV
SkJVVlZCUVRSSlFrUjNRWGRuWjBWTENrRnZTVUpCVVVOdFN6RmFRVWQ1Y1haME5qSm5NSGREZWts
S1IxcHBkbWhOVnk5a0wwSnpUM0pSYXpSWWJ6WllUM1ZKZVVwRWFFdEpNVVpuU2xkWGFtUUthblFy
TWs5YU1rNXpXVkpOWVU5clZGTXJZM012VVRseGVWTkhjRzVWZUd4UUwySkxlV1ZJYzJGUlJVOUla
VGhLV1dneGJFMUdkMm8yYlhKTmVtaDZlUW95V2toeU1rcGpPSEpIVFZCNlRHaElWVEJtSzJONGN5
ODNiMHBZVVdKVmQxcFBlR1pZTlU5eE1UQjNTVmRSZEhSdlUxQmFiMjlhTDBaM1ZIVm5UaXRpQ2xW
dk9GTTJTM2hxWlZCeWFHcExTbmhyTVd4YVZ6QjRTR3hwWXpGb1NtbERieXRVV2tRclNXVXZhM1pI
VEhsblpHeENZMk5tZEd4UFJYVnVkMjh6UlVrS2RYRk5kSG8zWVN0RWFrUk1MM3AwZDNOemFWaFFS

## Conclusion

Now we have created an Organization and Repository (monorepository) for the Flixtube application and configured Gitea Variables and Secrets for the Gitea Container Repository.

We we now ready to look at Gitea Actions for automating Continuous Integration and Continuous Delivery for the Flixtube application.